# Phase 1 - Locked Held-Out Retrieval Evaluation (Kaggle T4 x2)

Confirmatory retrieval-only evaluation of the locked Phase 1 winner on all 150 held-out articles (871 semantic-deduplicated resolved questions). This notebook does not rebuild an index and must not be used to retune the retrieval pipeline.

## 0. Locked configuration

The learned-sparse encoder runs on GPU 0 and the BGE-large reranker runs on GPU 1. `smoke` uses five development questions; `final` uses the complete held-out partition exactly once.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT = '6e56ce5643f162e9ef5ef5a6f0ab30d2c7626848'

DATA_ARTIFACT_REPO_ID = 'ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
DATA_ARTIFACT_REVISION = 'bb73e682f472933c212f2c6a3f9575c652b280fd'
DATA_ARTIFACT_FILENAME = 'artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
DATA_ARTIFACT_SHA256 = 'fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'

INDEX_REPO_ID = 'MatchaMacchiato/TextMining-Phase-1-Index'
INDEX_REVISION = '5ac4d9798ac96e09b4f06d7cc0d426510f4a7e39'
INDEX_RELATIVE_ROOT = Path('round1/sparse_bge_m3_sparse')
EXPECTED_INDEX_SHA256 = 'cfe11f27ec909f1146b2f6766d733c0e6c2351f84c60d62d72a5e76ce406c5ea'
EXPECTED_CHUNKS_SHA256 = '3a0f2aae08d0c978ae612c84692a3070ad2cb033935176b7c01eb7cbdc5d4498'
EXPECTED_TESTSET_SHA256 = '80f1b12acd6090aa353a89578c58c98e7d70d02dc4a271f2c724b370694c3e80'

RUN_MODE = 'smoke'  # smoke | final
EXECUTE = False     # Set True only after the preflight cell passes.
SEED = 42
DEVELOPMENT_ARTICLES = 50
TOP_K = 20
RERANK_TOP_N = 5
RERANKER_MODEL = 'BAAI/bge-reranker-large'
SMOKE_QUESTIONS = 5

KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_ROOT = KAGGLE_WORKING / 'Text-Mining---NewsQA-RAG'
WORK_ROOT = KAGGLE_WORKING / 'newsqa_phase1_heldout'
DATA_ROOT = WORK_ROOT / 'data'
INDEX_ROOT = WORK_ROOT / 'index_source'
RUNTIME_ROOT = WORK_ROOT / 'runtime'
EXPERIMENTS_ROOT = WORK_ROOT / 'experiments'
RESULTS_ROOT = WORK_ROOT / 'results'
LOG_ROOT = WORK_ROOT / 'logs'

## 1. Environment setup

Select **GPU T4 x2** and enable Internet in the Kaggle notebook settings. All referenced Hugging Face repositories are public, so no API secret is required.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

for path in [DATA_ROOT, INDEX_ROOT, RUNTIME_ROOT, EXPERIMENTS_ROOT, RESULTS_ROOT, LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
os.environ.update({
    'CUDA_VISIBLE_DEVICES': '0,1',
    'HF_HOME': str(KAGGLE_WORKING / 'hf_cache'),
    'TOKENIZERS_PARALLELISM': 'false',
    'OMP_NUM_THREADS': '2',
    'MKL_NUM_THREADS': '2',
    'PYTHONUNBUFFERED': '1',
})
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'fetch', '--depth=1', 'origin', REPO_COMMIT], cwd=PROJECT_ROOT, check=True, timeout=180)
subprocess.run(['git', 'checkout', '--detach', REPO_COMMIT], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], cwd=PROJECT_ROOT, check=True)

import pandas as pd
import torch
import yaml
from IPython.display import display

assert torch.cuda.is_available(), 'Enable the Kaggle GPU accelerator'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2, not a single-GPU accelerator'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip())

## 2. Download and verify locked artifacts

Only the BGE-M3 sparse index directory is downloaded from the Phase 1 index repository. The 1.8 GiB collection of unused dense/BM25 indexes is not downloaded. Chunks and the resolved test set come from the matching locked Phase 2 artifact.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

artifact_zip = Path(hf_hub_download(
    repo_id=DATA_ARTIFACT_REPO_ID, repo_type='dataset',
    revision=DATA_ARTIFACT_REVISION, filename=DATA_ARTIFACT_FILENAME,
))
assert sha256_file(artifact_zip) == DATA_ARTIFACT_SHA256
artifact_root = DATA_ROOT / 'locked-bge-m3-512-64-deduplicated-v2'
artifact_root.mkdir(parents=True, exist_ok=True)
required_members = [
    'bundle_manifest.json', 'chunks.jsonl', 'testset_resolved.jsonl',
    'deduplication/deduplicated.variant.json',
]
with zipfile.ZipFile(artifact_zip) as archive:
    for member in required_members:
        if not (artifact_root / member).exists():
            archive.extract(member, artifact_root)

index_snapshot = Path(snapshot_download(
    repo_id=INDEX_REPO_ID, repo_type='dataset', revision=INDEX_REVISION,
    allow_patterns=['manifest.json', f'{INDEX_RELATIVE_ROOT.as_posix()}/*'],
))
index_bundle = index_snapshot / INDEX_RELATIVE_ROOT
sparse_index = index_bundle / 'bge_m3_sparse.pkl'
chunks = artifact_root / 'chunks.jsonl'
testset = artifact_root / 'testset_resolved.jsonl'
assert sha256_file(sparse_index) == EXPECTED_INDEX_SHA256
assert sha256_file(chunks) == EXPECTED_CHUNKS_SHA256
assert sha256_file(testset) == EXPECTED_TESTSET_SHA256

upload_manifest = json.loads((index_snapshot / 'manifest.json').read_text())
records = {record['path']: record for record in upload_manifest['files']}
for path in sorted(index_bundle.rglob('*')):
    if path.is_file():
        relative = path.relative_to(index_snapshot).as_posix()
        assert relative in records and sha256_file(path) == records[relative]['sha256'], relative
print('Verified data and index artifacts')
print('Index source:', INDEX_REPO_ID, '@', INDEX_REVISION)
print('Sparse index:', round(sparse_index.stat().st_size / 2**20, 1), 'MiB')
print('Chunks:', sum(1 for _ in chunks.open()), '| Questions:', sum(1 for _ in testset.open()))

## 3. Rebind paths and verify the registered split

Changing a device or local path does not change the retrieval method. The runtime manifest records both immutable HF revisions and all content hashes.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / 'common'))
from newsqa_rag.evaluation.benchmark_io import stable_hash
from newsqa_rag.experiments import build_article_partitions

config = yaml.safe_load((index_bundle / 'config_sparse_bge_m3_sparse.yaml').read_text())
config['retrieval']['retriever'] = 'sparse'
config['retrieval']['top_k'] = TOP_K
config['retrieval']['sparse'].update({'method': 'bge-m3', 'model': 'BAAI/bge-m3', 'model_name': 'BAAI/bge-m3', 'device': 'cuda:0'})
config['retrieval']['reranker'].update({'enabled': True, 'type': 'cross-encoder', 'model': RERANKER_MODEL, 'top_n': RERANK_TOP_N, 'batch_size': 8, 'device': 'cuda:1'})
config_path = RUNTIME_ROOT / 'locked_phase1_heldout_config.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

variant = json.loads((index_bundle / 'variant_sparse_bge_m3_sparse.json').read_text())
variant['pipeline']['config_path'] = str(config_path)
variant['pipeline']['config_sha256'] = stable_hash(config)
variant['artifacts']['bm25'].update({'path': str(sparse_index), 'sha256': EXPECTED_INDEX_SHA256})
variant['artifacts']['chunks'].update({'path': str(chunks), 'sha256': EXPECTED_CHUNKS_SHA256})
variant['artifacts']['testset_resolved'].update({'path': str(testset), 'sha256': EXPECTED_TESTSET_SHA256})
variant['runtime_rebinding'] = {
    'index_repo_id': INDEX_REPO_ID, 'index_revision': INDEX_REVISION,
    'data_repo_id': DATA_ARTIFACT_REPO_ID, 'data_revision': DATA_ARTIFACT_REVISION,
    'sparse_device': 'cuda:0', 'reranker_device': 'cuda:1',
}
variant_path = RUNTIME_ROOT / 'locked_phase1_heldout_variant.json'
variant_path.write_text(json.dumps(variant, indent=2, sort_keys=True) + '\n', encoding='utf-8')

partitions = build_article_partitions({'resolved': testset}, DEVELOPMENT_ARTICLES, SEED, 'article_key')
development = partitions['partitions']['development']
heldout = partitions['partitions']['final_test']
assert len(development['article_ids']) == 50 and len(development['question_ids']['resolved']) == 281
assert len(heldout['article_ids']) == 150 and len(heldout['question_ids']['resolved']) == 871
(RUNTIME_ROOT / 'partitions_preflight.json').write_text(json.dumps(partitions, indent=2) + '\n')
display(pd.DataFrame([
    {'partition': 'development', 'articles': len(development['article_ids']), 'questions': len(development['question_ids']['resolved'])},
    {'partition': 'final_test', 'articles': len(heldout['article_ids']), 'questions': len(heldout['question_ids']['resolved'])},
]))

## 4. Execute smoke or final

Run `smoke` first. It touches only five development questions. Then restart with `RUN_MODE='final'`; that mode has no question limit and evaluates all 871 held-out questions.

In [ ]:
assert RUN_MODE in {'smoke', 'final'}
assert EXECUTE, 'After checking the hashes, split and GPU assignment, set EXECUTE=True'

experiment_id = f'phase1-locked-heldout-{RUN_MODE}'
partition = 'development' if RUN_MODE == 'smoke' else 'final_test'
spec = {
    'schema_version': 1,
    'experiment': {'id': experiment_id, 'name': f'Phase 1 locked {RUN_MODE}'},
    'output_dir': str(EXPERIMENTS_ROOT),
    'seed': SEED,
    'dataset': {
        'article_field': 'article_key',
        'development_articles': DEVELOPMENT_ARTICLES,
        'indexes': {
            'locked_bge_m3_512_64': {
                'config': str(config_path),
                'variant_manifest': str(variant_path),
                'testsets': {'resolved': str(testset)},
            }
        },
    },
    'fixed': {
        'retrieval_only': True, 'top_k': TOP_K,
        'rerank_top_n': RERANK_TOP_N, 'partition': partition,
    },
    'runs': [{
        'index': 'locked_bge_m3_512_64', 'variant': 'resolved',
        'retriever': 'sparse', 'reranker': 'cross-encoder',
        'reranker_model': RERANKER_MODEL,
    }],
    'runtime': {
        'max_attempts': 2, 'progress': True, 'warmup_queries': 3,
        **({'n_eval': SMOKE_QUESTIONS} if RUN_MODE == 'smoke' else {}),
    },
    'judge': {'enabled': False},
    'summary': {
        'metrics': [
            'retrieval.hit_rate@1', 'retrieval.hit_rate@3',
            'retrieval.hit_rate@5', 'retrieval.mrr@5',
            'retrieval.ndcg@5', 'retrieval.recall@5',
        ],
        'quality_metric': 'retrieval.mrr@5.mean',
        'latency_metric': 'latency.total.p50_ms',
    },
}
spec_path = RUNTIME_ROOT / f'{experiment_id}.yaml'
spec_path.write_text(yaml.safe_dump(spec, sort_keys=False), encoding='utf-8')

def run_logged(command, label):
    log_path = LOG_ROOT / f'{label}.log'
    print('$', ' '.join(map(str, command)), flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            [str(value) for value in command], cwd=PROJECT_ROOT, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            encoding='utf-8', errors='replace', bufsize=1,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return log_path

started = time.time()
run_logged([sys.executable, '-u', 'scripts/run_experiment.py', spec_path], f'{RUN_MODE}_experiment')
experiment_root = EXPERIMENTS_ROOT / experiment_id
run_logged([sys.executable, '-u', 'scripts/summarize_experiments.py', experiment_root], f'{RUN_MODE}_summary')
print('Elapsed minutes:', round((time.time() - started) / 60, 1))

## 5. Validate, visualize and export

A final run is accepted only with 150 articles, 871 expected questions, no missing records and 100% successful retrieval coverage.

In [ ]:
comparison_path = experiment_root / 'comparison.csv'
comparison = pd.read_csv(comparison_path)
assert len(comparison) == 1
row = comparison.iloc[0]
expected = SMOKE_QUESTIONS if RUN_MODE == 'smoke' else 871
assert int(row['coverage.expected']) == expected
assert int(row['coverage.missing']) == 0
assert int(row['coverage.failed']) == 0
assert float(row['coverage.success_rate']) == 1.0

columns = [
    'variant', 'partition', 'coverage.expected', 'coverage.success_rate',
    'retrieval.hit_rate@1.mean', 'retrieval.hit_rate@3.mean',
    'retrieval.hit_rate@5.mean', 'retrieval.mrr@5.mean',
    'retrieval.ndcg@5.mean', 'retrieval.recall@5.mean',
    'latency.retrieve_ms.p50_ms', 'latency.rerank_ms.p50_ms',
    'latency.total.p50_ms', 'latency.total.p95_ms',
]
display(comparison[[column for column in columns if column in comparison.columns]])

import matplotlib.pyplot as plt
metrics = {
    'Hit@1': row['retrieval.hit_rate@1.mean'],
    'Hit@3': row['retrieval.hit_rate@3.mean'],
    'Hit@5': row['retrieval.hit_rate@5.mean'],
    'MRR@5': row['retrieval.mrr@5.mean'],
    'NDCG@5': row['retrieval.ndcg@5.mean'],
    'Recall@5': row['retrieval.recall@5.mean'],
}
figure, axis = plt.subplots(figsize=(9, 4.5))
axis.bar(metrics.keys(), metrics.values(), color=['#28666e', '#2f7f72', '#3a9275', '#d17b46', '#b85c5c', '#6b7280'])
axis.set_ylim(0, 1); axis.set_ylabel('Score'); axis.set_title(f'Phase 1 locked retrieval - {RUN_MODE}')
for position, value in enumerate(metrics.values()): axis.text(position, value + 0.015, f'{value:.3f}', ha='center')
figure.tight_layout()
figure_path = RESULTS_ROOT / f'{RUN_MODE}_retrieval_metrics.png'
figure.savefig(figure_path, dpi=200); plt.show(); plt.close(figure)

shutil.copy2(comparison_path, RESULTS_ROOT / f'{RUN_MODE}_comparison.csv')
shutil.copy2(experiment_root / 'comparison.json', RESULTS_ROOT / f'{RUN_MODE}_comparison.json')
run_directories = [path.parent for path in experiment_root.glob('*/report.json')]
assert len(run_directories) == 1
run_directory = run_directories[0]
for filename in ['report.json', 'summary.txt', 'deterministic_scores.jsonl', 'predictions.jsonl', 'retrievals.jsonl', 'run_manifest.json', 'environment.json']:
    source = run_directory / filename
    if source.exists(): shutil.copy2(source, RESULTS_ROOT / filename)

protocol = {
    'schema_version': 1, 'run_mode': RUN_MODE, 'confirmatory_only': RUN_MODE == 'final',
    'winner_locked_before_heldout': True, 'no_post_heldout_reselection': True,
    'dataset': {'articles': 150 if RUN_MODE == 'final' else 50, 'questions': expected, 'variant': 'resolved'},
    'retrieval': {'method': 'bge-m3 learned sparse', 'top_k': TOP_K},
    'reranker': {'model': RERANKER_MODEL, 'top_n': RERANK_TOP_N},
    'chunking': {'strategy': 'recursive', 'chunk_size': 512, 'chunk_overlap': 64},
    'sources': {
        'repository_commit': REPO_COMMIT,
        'index_repo_id': INDEX_REPO_ID, 'index_revision': INDEX_REVISION,
        'index_sha256': EXPECTED_INDEX_SHA256,
        'data_repo_id': DATA_ARTIFACT_REPO_ID, 'data_revision': DATA_ARTIFACT_REVISION,
        'chunks_sha256': EXPECTED_CHUNKS_SHA256, 'testset_sha256': EXPECTED_TESTSET_SHA256,
    },
}
(RESULTS_ROOT / f'{RUN_MODE}_protocol.json').write_text(json.dumps(protocol, indent=2, sort_keys=True) + '\n')

archive_path = KAGGLE_WORKING / f'phase1_locked_heldout_{RUN_MODE}_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for source_root in [RESULTS_ROOT, RUNTIME_ROOT, LOG_ROOT]:
        for path in source_root.rglob('*'):
            if path.is_file(): archive.write(path, path.relative_to(WORK_ROOT))
print('Downloadable results:', archive_path, round(archive_path.stat().st_size / 2**20, 1), 'MiB')

## Interpretation rule

The held-out metrics estimate robustness of the already locked retrieval pipeline. They must be reported even if lower than development results. Do not change the retriever, reranker, chunking, `top_k`, or `top_n` after reading them.